# 📝 과제 02: 온라인 쇼핑 상담 모델 평가와 일반 능력 퇴행 줄이기

## 만들 결과

교안 02의 과제입니다. 과제 01이 에폭마다 저장한 온라인 쇼핑 상담 체크포인트와 학습 전 모델(Base)을 교안 02와 같은 지표 일곱 개로 평가해 교안 02와 같은 규칙으로 Tuned를 고르고, 퇴행을 줄인 후보인 가중치 보간 모델까지 지표를 MLflow에 기록하고 결정 테이블 하나로 비교합니다. 마지막으로 고른 Tuned를 최종 모델 폴더에 저장합니다.

**만들 것**
- 모델마다 지표 일곱 개를 모은 사전 `model_metrics`와 테이블 `forgetting_table`(Base, 에폭별 체크포인트)
- 규칙으로 고른 Tuned의 갱신 횟수 `tuned_step`과 에폭 수 `tuned_epoch`
- Base와 마지막 체크포인트를 α 두 개(0.5, 0.75)로 섞은 보간 모델의 지표
- MLflow run에 더한 지표와, 서비스에 쓸 모델을 고르는 결정 테이블 `summary_table`
- 고른 Tuned 체크포인트의 모델과 토크나이저를 저장한 최종 모델 폴더 `output/commerce_sft_final`

## 준비: 데이터와 제공 코드

- **먼저 할 일:** 과제 01을 끝까지 실행해 체크포인트 `output/commerce_epochs/checkpoint-<갱신 횟수>` 네 개(갱신 횟수 34, 68, 102, 136)와 MLflow run `epochs4_lr_5e-6`를 만들어 둡니다. 교안과 같은 RunPod GPU 파드(GPU 메모리 24GB 이상)에서 실행합니다.
- **API 키와 호출 수:** 판사는 교안 02와 같은 Jev(`jev-1.13.0`, `TYPESAFE_MODEL`로 바꿈)라 `.env`에 `TYPESAFE_API_KEY`가 필요합니다. 모델 하나에 90번, 모델 7개에 모두 약 630번 부릅니다.
- **자료(`data/과제/`):** `commerce_train.jsonl`, `commerce_val.jsonl`(과제 01과 같음), `commerce_eval.jsonl`(평가셋 20문항, 재현 13·적용 7), `commerce_out_of_scope_eval.jsonl`(범위 밖 질문 7개). 일반 질문과 일반 문단은 교안 02와 같은 `data/본실습/` 파일입니다.
- **제공 코드:** 판사와 채점·지표 함수는 교안 02와 같습니다(`answer_and_judge(model, rows, system_prompt)`, `perplexity(model, texts)`, `evaluate_model(model)`, `table_metrics(tables)`, `METRIC_NAMES`). `evaluate_model`은 1번에서 만드는 `train_sample_rows`를 씁니다.
- **교안 02와 다른 점:** 평가셋 적용 문항이 7개, 범위 밖 질문이 7개라 한 문항이 적용 정답률을 1/7, 거절률을 1/7만큼 바꾸고, 참일 확률이 0.5 근처인 판정 문장은 작은 차이로 통과와 실패가 갈립니다. 고르는 규칙은 그대로 적용하되, 한두 문항 차이는 조심해서 읽습니다.
- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행합니다. 뒤 문항이 앞 문항의 변수를 쓰므로 차례로 풉니다. 자가채점은 판사 점수 값이 아니라 테이블의 모양과 계산 관계를 확인합니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저: 학습 전후 모델이 같은 토크나이저를 씁니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("대화 끝 토큰:", tokenizer.eos_token, "| 패딩 토큰:", tokenizer.pad_token)

In [ ]:
# 지난 시간 과제 데이터의 시스템 프롬프트와 범위 밖 거절 답입니다(학습 데이터의 messages와 글자까지 같습니다)
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."
REFUSAL_ANSWER_COMMERCE = "이 상담 챗봇이 다루는 전자상거래법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_replies(model, questions, system_prompt, max_new_tokens=256):
    """질문 목록을 32개씩 묶어 한 번에 생성하고, 질문 순서대로 새로 만든 답 목록을 돌려줍니다.

    system_prompt=None 이면 시스템 메시지 없이 묻습니다(일반 능력 점검용).
    """
    # 여러 질문을 한 번에 생성하려면 짧은 질문 앞을 패딩으로 채웁니다. 답은 오른쪽 끝에 이어 쓰므로 패딩을 왼쪽에 둡니다
    tokenizer.padding_side = "left"

    answers = []
    for start in range(0, len(questions), 32):
        # 1) 질문마다 학습 데이터와 같은 chat template으로 감쌉니다
        prompts = []
        for question in questions[start:start + 32]:
            messages = []
            if system_prompt is not None:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": question})
            prompts.append(tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,  # 끝에 assistant 머리를 붙여 모델이 답을 이어 쓰게 합니다
                enable_thinking=False,       # 빈 생각 블록을 붙여 추론 과정 없이 바로 답합니다
            ))
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,  # 길이가 다른 질문을 패딩으로 맞춰 한 텐서로 묶습니다
        ).to(model.device)

        # 2) 대화 끝 토큰까지 또는 max_new_tokens 까지 답을 만듭니다
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,         # 새로 만들 토큰 수의 상한
                do_sample=False,                       # 그리디 생성: 같은 질문에 같은 답
                eos_token_id=END_OF_TURN_ID,           # 대화 끝 토큰 <|im_end|>에서 멈춥니다
                pad_token_id=tokenizer.pad_token_id,   # 넘기지 않으면 패딩 토큰 경고가 나옵니다
            )

        # 3) 입력 길이만큼 잘라 새로 만든 토큰만 글자로 바꿉니다(먼저 끝난 답 뒤에 붙은 패딩은 skip_special_tokens 로 빠집니다)
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            answers.append(tokenizer.decode(new_ids, skip_special_tokens=True))

    # 학습기가 쓰는 오른쪽 패딩으로 되돌립니다
    tokenizer.padding_side = "right"
    return answers


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def to_prompt_completion(row):
    """messages를 앞부분(system이 있으면 system, user)과 배울 답(마지막 assistant)으로 나눕니다."""
    prompt = row["messages"][:-1]      # 마지막 답 앞의 메시지(system, user)
    completion = row["messages"][-1:]  # 마지막 assistant 메시지 하나
    return {"prompt": prompt, "completion": completion}

In [ ]:
# 학습 전 모델(Base)은 허브에서 불러옵니다(과제 01과 같은 float32). 체크포인트는 4번에서 하나씩 불러옵니다
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
print("Base:", MODEL_NAME)

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("commerce-sft")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

In [ ]:
# 학습, 검증 데이터와 평가에 쓰는 네 가지 자료를 읽습니다
train_rows = load_jsonl(data_dir / "과제" / "commerce_train.jsonl")
val_rows = load_jsonl(data_dir / "과제" / "commerce_val.jsonl")
eval_rows = load_jsonl(data_dir / "과제" / "commerce_eval.jsonl")                       # 쇼핑 상담 평가셋: question_type 은 재현, 적용
out_of_scope_rows = load_jsonl(data_dir / "과제" / "commerce_out_of_scope_eval.jsonl")  # 범위 밖 질문
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")              # 일반 질문과 정답 답변(교안 02와 같은 파일)
general_texts = json.loads((data_dir / "본실습" / "general_texts.json").read_text(encoding="utf-8"))  # 일반 문단(perplexity)

print("학습:", len(train_rows), "| 검증:", len(val_rows), "| 평가셋:", len(eval_rows), "| 범위 밖:", len(out_of_scope_rows),
      "| 일반 질문:", len(general_rows), "| 일반 문단:", len(general_texts))
print("평가셋 종류별:", pd.DataFrame(eval_rows)["question_type"].value_counts().to_dict())

아래 셀은 교안 02와 같은 판사, 채점 함수, 지표 함수입니다. 실행만 하고 넘어갑니다.

In [ ]:
import os

from dotenv import load_dotenv

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 채웁니다
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # 사용 기록을 외부로 보내지 않음(deepeval을 불러오기 전에 설정)
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig
from deepeval.metrics import JevEval
from deepeval.metrics.jev_eval import Noul
from deepeval.test_case import LLMTestCase, SingleTurnParams

# 판정 결과가 모델 버전에 따라 달라지지 않게 Jev 버전을 고정합니다
jev_model_name = os.getenv("TYPESAFE_MODEL", "jev-1.13.0")
print("Jev 모델:", jev_model_name)
print("Jev 키 읽음:", os.getenv("TYPESAFE_API_KEY") is not None)

In [ ]:
# 두 판사가 함께 쓰는 판정 문장. Jev가 문장마다 참일 확률을 돌려줍니다
BASE_STATEMENTS = [
    # 1. 질문에 직접 답했는가
    Noul("actual_output은 input의 질문에 직접 답한다."),
    # 2. 사실과 결론이 정답 답변과 같은가(표현이 달라도 같은 정보면 같다고 봅니다)
    Noul("actual_output의 사실(숫자, 기간, 조건, 이름)과 결론이 expected_output과 일치한다. 표현이 달라도 같은 정보를 전하면 일치로 본다."),
    # 3. 질문과 관계없는 지어낸 내용을 덧붙이지 않았는가
    Noul("actual_output은 input의 질문과 관계없는 지어낸 내용을 덧붙이지 않는다."),
    # 4. 같은 말을 되풀이하지 않았는가
    Noul("actual_output은 같은 문장이나 구절을 되풀이하지 않는다."),
]

# 일반 질문과 학습 데이터 표본용: 질문, 모델의 답, 정답 답변
general_metric = JevEval(
    name="정확성",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],  # 판정 문장에서 가리키는 필드
    questions=BASE_STATEMENTS,
    system_one_model=jev_model_name,
    strict_mode=True,  # 모든 문장의 참일 확률이 0.5 이상이어야 통과
    async_mode=True,  # Jev 호출을 비동기로 보냄(DeepEval 기본값)
)

# 쇼핑 상담 질문용: 근거 조문(context)도 함께 봅니다
law_metric = JevEval(
    name="정확성(근거 조문)",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    questions=BASE_STATEMENTS + [
        # 5. 정답 답변에 없는 말은 근거 조문과 어긋나지 않는가
        Noul("actual_output이 expected_output에 없는 내용을 말할 때, 그 내용은 context(근거 조문)와 모순되지 않는다."),
    ],
    system_one_model=jev_model_name,
    strict_mode=True,
    async_mode=True,
)


# 판사를 동시에 부르는 설정(LLM 판사 단원과 같은 방법, 동시에 보내는 상한은 교안 02와 같은 64)
judge_config = {
    # 동시에 보낼 문항 수 상한. 반 전체가 같은 Jev 키(TYPESAFE_API_KEY)를 쓰면 429(요청 한도 초과)가 날 수 있어, 429가 나면 값을 줄입니다
    "async_config": AsyncConfig(max_concurrent=64),
    "display_config": DisplayConfig(print_results=False, show_indicator=False),  # 긴 기본 출력을 끕니다
}


def judge_answers(rows, answers):
    """판사가 답 목록을 동시에 채점해, 문항 순서대로 점수, 통과, 이유 열을 가진 테이블을 돌려줍니다.

    행에 근거 조문(law_text)이 있으면 근거 조문 기준으로, 없으면 일반 기준으로 채점합니다.
    """
    law_cases = []
    general_cases = []
    for index in range(len(rows)):
        row = rows[index]
        if "law_text" in row:
            law_cases.append(LLMTestCase(
                name=str(index),                          # 결과를 문항 순서로 되돌리는 번호(동시 채점은 끝나는 순서가 섞입니다)
                input=row["question"],                    # 질문
                actual_output=answers[index],             # 채점할 모델의 답
                expected_output=row["reference_answer"],  # 정답 답변
                context=[row["law_text"]],                # 근거 조문 원문
            ))
        else:
            general_cases.append(LLMTestCase(
                name=str(index),                          # 결과를 문항 순서로 되돌리는 번호(동시 채점은 끝나는 순서가 섞입니다)
                input=row["question"],                    # 질문
                actual_output=answers[index],             # 채점할 모델의 답
                expected_output=row["reference_answer"],  # 정답 답변
            ))

    judged = {}
    for test_cases, metric in [(law_cases, law_metric), (general_cases, general_metric)]:
        if len(test_cases) == 0:
            continue
        result = evaluate(test_cases=test_cases, metrics=[metric], **judge_config)
        for test_result in result.test_results:
            metric_data = test_result.metrics_data[0]  # 지표를 하나만 넘겼습니다
            # 점수는 모든 문장이 참이면 1, 아니면 0입니다. 이유에는 문장마다 참일 확률 P(yes)가 나옵니다
            judged[test_result.name] = (metric_data.score, metric_data.success, metric_data.reason)

    judged_list = []
    for index in range(len(rows)):
        judged_list.append(judged[str(index)])
    return pd.DataFrame(judged_list, columns=["점수", "통과", "이유"])

In [ ]:
def has_keywords(answer, keywords):
    """키워드가 답에 모두 들어 있으면 True를 돌려줍니다."""
    for keyword in keywords:
        if keyword not in answer:
            return False
    return True


def answer_and_judge(model, rows, system_prompt):
    """rows의 질문에 답을 한꺼번에 생성하고 판사가 동시에 채점해, 질문마다 한 행씩 모읍니다."""
    questions = []
    for row in rows:
        questions.append(row["question"])
    answers = generate_replies(model, questions, system_prompt)

    result_rows = []
    for row, answer in zip(rows, answers):
        result_rows.append({"질문": row["question"], "종류": row.get("question_type", ""), "답변": answer,
                            "거절": has_keywords(answer, ["답할 수 없"])})  # 거절률과 과잉 거절률에 씁니다
    result_table = pd.DataFrame(result_rows)

    # 범위 밖 질문은 정답 답변이 없어 판사를 부르지 않습니다
    if "reference_answer" in rows[0]:
        result_table = pd.concat([result_table, judge_answers(rows, answers)], axis=1)
    return result_table

In [ ]:
import math


def perplexity(model, texts):
    """문단 목록 전체의 perplexity: 예측한 토큰 전체의 교차 엔트로피 평균에 exp를 합니다."""
    total_loss = 0.0
    total_tokens = 0
    for text in texts:
        input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)

        with torch.no_grad():
            loss = model(
                input_ids,
                labels=input_ids,  # 입력을 정답으로 주면 다음 토큰을 하나씩 예측한 손실의 평균을 돌려줍니다
            ).loss.item()

        # 첫 토큰은 앞 토큰이 없어 예측하지 않으므로 예측한 토큰 수는 길이 - 1. 평균 * 개수로 이 문장의 손실 합을 만듭니다
        n_predicted = input_ids.shape[1] - 1
        total_loss = total_loss + loss * n_predicted
        total_tokens = total_tokens + n_predicted

    # 전체 손실 합 / 전체 예측 토큰 수 = 평균 손실, 여기에 exp
    return math.exp(total_loss / total_tokens)

In [ ]:
# 지표 이름(MLflow 에 기록할 이름)과 테이블에 보일 이름. 학습 데이터 정답률을 맨 앞에 둡니다
METRIC_NAMES = {"train_acc": "학습 데이터 정답률", "domain_recall_acc": "도메인 정답률(재현)", "domain_apply_acc": "도메인 정답률(적용)",
                "over_refusal_rate": "과잉 거절률", "refusal_rate": "거절률", "general_acc": "일반 질문 정답률", "general_ppl": "일반 perplexity"}


def evaluate_model(model):
    """모델 하나에 앞의 평가를 모두 되풀이해 채점 테이블 네 개와 perplexity 를 사전으로 돌려줍니다.

    판사 호출 수 = 표본 수 + 평가 문항 수 + 일반 질문 수
    """
    return {
        "train": answer_and_judge(model, train_sample_rows, SYSTEM_PROMPT_COMMERCE),
        "domain": answer_and_judge(model, eval_rows, SYSTEM_PROMPT_COMMERCE),
        "refusal": answer_and_judge(model, out_of_scope_rows, SYSTEM_PROMPT_COMMERCE),
        "general": answer_and_judge(model, general_rows, None),
        "ppl": perplexity(model, general_texts),
    }


def table_metrics(tables):
    """evaluate_model 이 돌려준 사전으로 지표 일곱 개를 계산합니다."""
    domain_table = tables["domain"]
    return {
        "train_acc": tables["train"]["통과"].mean(),
        "domain_recall_acc": domain_table[domain_table["종류"] == "재현"]["통과"].mean(),
        "domain_apply_acc": domain_table[domain_table["종류"] == "적용"]["통과"].mean(),
        "over_refusal_rate": domain_table["거절"].mean(),
        "refusal_rate": tables["refusal"]["거절"].mean(),
        "general_acc": tables["general"]["통과"].mean(),
        "general_ppl": tables["ppl"],
    }

## 1. 학습 데이터 정답률과 도메인 정답률(Base)

**배경**: 교안 02의 2-1 평가셋과 학습 데이터 표본, 2-2 답 생성과 판사 채점과 같은 방법입니다. 학습한 질문에도 답하지 못하는 모델은 학습 데이터에 없는 질문에도 맞게 답하지 못하므로, 학습 데이터 표본의 정답률을 먼저 봅니다.

**요구사항**
- (1) `train_rows` 가운데 `answer`에 거절 키워드 `"답할 수 없"`가 없는 행만 파일 순서대로 **`answer_rows`** 에 담습니다(거절은 2번에서 범위 밖 질문으로 따로 계산합니다).
- (2) 뽑는 간격 **`sample_gap`**(`len(answer_rows) // 40`)마다 한 행씩 앞에서부터 뽑은 앞 40행으로, 키가 `"question"`, `"reference_answer"`(학습한 답 `answer`)인 사전 리스트 **`train_sample_rows`** 를 만듭니다.
- (3) `answer_and_judge`로 Base의 답을 쇼핑 상담 시스템 프롬프트(`SYSTEM_PROMPT_COMMERCE`)로 채점한 테이블을, 학습 데이터 표본은 **`base_train_table`**, 평가셋 `eval_rows`는 **`base_domain_table`** 에 담습니다.

**확인 기준**
- `answer_rows`는 128행이고, `train_sample_rows`의 키·행 수·질문 순서가 자가채점 셀의 지문과 같습니다.
- 두 채점 테이블은 입력 행마다 한 행이고 `"통과"`는 점수가 1(판정 문장이 모두 참)인지와 같습니다. 판사 점수 값 자체는 보지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 학습 데이터 표본 셀과 Base 채점 셀과 같습니다. 거절 답 행을 거른 뒤 슬라이싱 간격으로 고르게 뽑고, 채점은 answer_and_judge 한 번씩이면 됩니다.

세부구현:
1. for 문과 if 문으로 키워드가 answer에 없는 행만 리스트에 더합니다.
2. 리스트[::간격]은 간격마다 하나씩 고르고, 뒤에 [:개수]를 붙이면 앞에서 그 개수만 남깁니다.
3. answer_and_judge의 셋째 인자가 시스템 프롬프트입니다.
```

</details>

In [ ]:
# [작성] (1) 거절 답 행을 빼고, (2) 학습 데이터 표본을 고르게 뽑은 뒤, (3) Base의 답을 판사로 채점하세요.

# (1) 거절 답 행은 뺍니다(거절은 2번에서 범위 밖 질문으로 따로 계산합니다)
answer_rows = []
for row in train_rows:
    ...

# (2) 학습 데이터 표본: 남은 행 수를 40으로 나눈 간격마다 한 행씩, 앞에서 끝까지 고르게 뽑습니다
sample_gap = ...
train_sample_rows = []
for row in ...:
    # 학습한 답(answer)을 정답 답변으로 씁니다. 근거 조문 원문(law_text)이 없어 판사는 일반 기준으로 채점합니다
    ...
print("학습 행 수:", len(train_rows), "| 거절 답을 뺀 행 수:", len(answer_rows), "| 뽑는 간격:", sample_gap, "| 표본 수:", len(train_sample_rows))

# (3) 쇼핑 상담 시스템 프롬프트로 Base의 답을 만들고 채점합니다. 판사 호출 수 = 표본 수 + 평가 문항 수
base_train_table = ...
base_domain_table = ...

# 평가셋의 앞 다섯 문항: 답, 판정, 판정 이유
display(base_domain_table[["종류", "질문", "답변", "통과", "이유"]].head())

# 통과 비율은 True를 1, False를 0으로 본 평균입니다. 통과 수 / 문항 수로도 봅니다
print(f"학습 데이터 정답률: {base_train_table['통과'].mean():.3f} ({base_train_table['통과'].sum()}/{len(base_train_table)})")
for question_type in ["재현", "적용"]:
    base_part = base_domain_table[base_domain_table["종류"] == question_type]
    print(f"도메인 정답률({question_type}): {base_part['통과'].mean():.3f} ({base_part['통과'].sum()}/{len(base_part)})")

In [ ]:
# [자가채점] 표본 행, 두 채점 테이블의 열과 행, 통과와 점수의 관계를 확인합니다(판사 점수 값 자체는 보지 않습니다)
import hashlib

judge_columns = ["질문", "종류", "답변", "거절", "점수", "통과", "이유"]

# (1) 거절 답 행을 뺀 학습 행
assert isinstance(answer_rows, list) and len(answer_rows) == 128, f"answer_rows가 {len(answer_rows)}행입니다. train_rows에서 answer에 \"답할 수 없\"가 없는 행 128개를 담으세요."
for row in answer_rows:
    assert "답할 수 없" not in row["answer"], f"{row['id']}는 거절 답 행입니다. answer_rows에서 빼세요."

# (2) 학습 데이터 표본: 행 수, 키, 질문 순서의 지문(sha256 앞 16자리), 정답 답변
answer_by_question = {}
for row in train_rows:
    answer_by_question[row["question"]] = row["answer"]
assert isinstance(train_sample_rows, list) and len(train_sample_rows) == 40, f"train_sample_rows가 {len(train_sample_rows)}행입니다. 간격마다 뽑은 앞 40행을 담으세요."
sample_questions = []
for row in train_sample_rows:
    assert list(row) == ["question", "reference_answer"], f"train_sample_rows의 행 키가 {list(row)}입니다. 키는 question, reference_answer 두 개입니다."
    assert row["reference_answer"] == answer_by_question[row["question"]], "reference_answer는 그 학습 행의 answer(학습한 답)입니다."
    sample_questions.append(row["question"])
assert hashlib.sha256("\n".join(sample_questions).encode("utf-8")).hexdigest()[:16] == "37ab7bf0d6c49a35", "train_sample_rows의 행이나 순서가 다릅니다. answer_rows[::sample_gap]의 앞 40행을 파일 순서대로 담으세요."

# (3) 두 채점 테이블
for table_name, table, rows in [("base_train_table", base_train_table, train_sample_rows), ("base_domain_table", base_domain_table, eval_rows)]:
    assert isinstance(table, pd.DataFrame), f"{table_name}은 answer_and_judge가 돌려준 테이블입니다. `...`를 채웠는지 확인하세요."
    assert list(table.columns) == judge_columns, f"{table_name}의 열이 {list(table.columns)}입니다. answer_and_judge(base_model, 행, SYSTEM_PROMPT_COMMERCE)의 반환값을 그대로 담으세요."
    assert len(table) == len(rows), f"{table_name}은 입력 행마다 한 행입니다."
    for row, question, score, passed in zip(rows, table["질문"], table["점수"], table["통과"]):
        assert question == row["question"], f"{table_name}의 질문이 입력 행과 다릅니다."
        assert 0 <= score <= 1 and bool(passed) == (score == 1), f"{table_name}의 통과 열이 점수 1인지와 다릅니다."
for row, question_type in zip(eval_rows, base_domain_table["종류"]):
    assert question_type == row["question_type"], "base_domain_table의 종류가 eval_rows의 question_type과 다릅니다."
print("✅ 통과!")

## 2. 범위 밖 질문 거절률과 과잉 거절률(Base)

**배경**: 교안 02의 2-4 범위 밖 질문 거절과 과잉 거절과 같은 방법입니다. 거절 여부는 판사 없이 거절 키워드(`"답할 수 없"`)로 셉니다. 거절률은 높을수록, 답할 수 있는 질문을 거절한 과잉 거절률은 낮을수록 좋습니다.

**요구사항**
- (1) `answer_and_judge`로 `out_of_scope_rows`를 쇼핑 상담 시스템 프롬프트로 물은 테이블을 **`base_refusal_table`** 에 담습니다(정답 답변이 없어 판사는 부르지 않습니다).
- (2) 범위 밖 테이블의 `"거절"` 평균을 **`base_refusal_rate`**, 1번 도메인 채점 테이블의 `"거절"` 평균을 **`base_over_refusal_rate`** 에 담습니다.

**확인 기준**
- `base_refusal_table`은 범위 밖 질문마다 한 행이고, `"거절"`은 `has_keywords(답변, ["답할 수 없"])`와 같습니다.
- 두 비율은 그 테이블에서 거절한 답의 비율과 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 범위 밖 질문 셀과 같습니다. 범위 밖 질문도 서비스와 같은 시스템 프롬프트로 묻습니다.

세부구현:
1. answer_and_judge는 행에 reference_answer가 없으면 점수, 통과, 이유 열을 만들지 않습니다.
2. 거절 열은 참·거짓이라 mean()이 거절한 비율입니다.
```

</details>

In [ ]:
# [작성] (1) 범위 밖 질문을 Base에 묻고, (2) 거절률과 과잉 거절률을 계산하세요.

# (1) 서비스와 같은 쇼핑 상담 시스템 프롬프트로 묻습니다(정답 답변이 없어 판사는 부르지 않습니다)
base_refusal_table = ...
display(base_refusal_table[["질문", "답변", "거절"]])

# (2) 거절률은 범위 밖 질문의 거절 비율, 과잉 거절률은 1번 평가셋 답의 거절 비율입니다
base_refusal_rate = ...
base_over_refusal_rate = ...
print("거절률:", round(base_refusal_rate, 3), "| 과잉 거절률:", round(base_over_refusal_rate, 3))

In [ ]:
# [자가채점] 범위 밖 테이블의 열과 행, 거절 열, 거절률과 과잉 거절률을 확인합니다
assert isinstance(base_refusal_table, pd.DataFrame), "base_refusal_table은 answer_and_judge가 돌려준 테이블입니다. `...`를 채웠는지 확인하세요."
assert list(base_refusal_table.columns) == ["질문", "종류", "답변", "거절"], f"base_refusal_table의 열이 {list(base_refusal_table.columns)}입니다. answer_and_judge(base_model, out_of_scope_rows, SYSTEM_PROMPT_COMMERCE)의 반환값을 담으세요."
assert len(base_refusal_table) == len(out_of_scope_rows), "base_refusal_table은 out_of_scope_rows의 질문마다 한 행입니다."
for answer, refused in zip(base_refusal_table["답변"], base_refusal_table["거절"]):
    assert refused == has_keywords(answer, ["답할 수 없"]), "base_refusal_table의 거절 열이 거절 키워드 포함 여부와 다릅니다."

# 비율 * 답의 수 = 거절한 답의 수
for value_name, value, table in [("base_refusal_rate", base_refusal_rate, base_refusal_table), ("base_over_refusal_rate", base_over_refusal_rate, base_domain_table)]:
    refused_count = 0
    for refused in table["거절"]:
        refused_count = refused_count + int(refused)
    assert 0 <= value <= 1 and abs(value * len(table) - refused_count) < 1e-6, f"{value_name}가 {value}입니다. 답 {len(table)}개 가운데 거절 {refused_count}개의 비율이어야 합니다. 거절률은 범위 밖 테이블, 과잉 거절률은 1번 도메인 채점 테이블에서 셉니다."
print("✅ 통과!")

## 3. 일반 능력과 지표 일곱 개(Base)

**배경**: 교안 02의 3-1 일반 질문, 3-2 일반 문단 perplexity, 3-3 지표 일곱 개 모으기와 같은 방법입니다. 쇼핑 상담 프롬프트 아래에서는 상식 질문도 거절이 맞으므로, 일반 질문은 시스템 프롬프트 없이 묻습니다.

**요구사항**
- (1) `answer_and_judge`로 `general_rows`를 시스템 프롬프트 없이(`None`) 채점한 테이블을 **`base_general_table`** 에, `perplexity(base_model, general_texts)`를 **`base_general_ppl`** 에 담습니다.
- (2) 1~3번의 채점 테이블과 perplexity를 `evaluate_model`이 돌려주는 모양(키 `"train"`, `"domain"`, `"refusal"`, `"general"`, `"ppl"`)의 사전 **`base_tables`** 로 모읍니다. 답을 다시 생성하지 않습니다.
- (3) `table_metrics(base_tables)`를 키 `"Base"`로 담은 사전 **`model_metrics`** 와, `pd.DataFrame(model_metrics).T.rename(columns=METRIC_NAMES)`로 **`forgetting_table`** 을 만듭니다.

**확인 기준**
- `base_general_table`은 일반 질문마다 한 행이고 열이 1번 채점 테이블과 같습니다. `base_general_ppl`은 1보다 큰 유한한 값입니다.
- `base_tables`의 키는 위 다섯 개, `model_metrics["Base"]`의 키는 `METRIC_NAMES`의 일곱 개이고, `forgetting_table`은 행 `"Base"`와 열 일곱 개입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 일반 질문, 일반 문단 perplexity, 지표 일곱 개 모으기 셀과 같습니다.

세부구현:
1. answer_and_judge의 셋째 인자에 None을 넣으면 시스템 메시지 없이 묻습니다.
2. 사전의 값에 앞 문항의 테이블 변수를 그대로 넣습니다.
3. pd.DataFrame(사전)은 바깥 키가 열이 되므로 .T로 뒤집어 모델을 행으로 둡니다.
```

</details>

In [ ]:
# [작성] (1) 일반 질문을 시스템 프롬프트 없이 채점하고 perplexity를 계산한 뒤, (2) 채점 결과를 사전으로 모아 (3) 지표 테이블을 만드세요.

# (1) 시스템 프롬프트 없이 묻고 채점합니다. 판사 호출 수 = 일반 질문 수
base_general_table = ...
display(base_general_table[["질문", "답변", "통과"]])
print("일반 질문 정답률:", round(base_general_table["통과"].mean(), 3))

# Base의 일반 문단 perplexity
base_general_ppl = ...
print("Base 일반 perplexity:", round(base_general_ppl, 1))

# (2) Base는 앞에서 만든 채점 테이블을 evaluate_model 과 같은 모양의 사전으로 모읍니다(답을 다시 생성하지 않습니다)
base_tables = ...

# (3) 지표 일곱 개를 모델마다 한 행에 모읍니다. 뒤 문항에서 다른 모델도 이 사전에 더합니다
model_metrics = ...
forgetting_table = ...
display(forgetting_table.round(3))

In [ ]:
# [자가채점] 일반 질문 테이블, perplexity, 채점 사전, 지표 사전과 테이블을 확인합니다(판사 점수 값 자체는 보지 않습니다)
judge_columns = ["질문", "종류", "답변", "거절", "점수", "통과", "이유"]
assert isinstance(base_general_table, pd.DataFrame), "base_general_table은 answer_and_judge가 돌려준 테이블입니다. `...`를 채웠는지 확인하세요."
assert list(base_general_table.columns) == judge_columns and len(base_general_table) == len(general_rows), "base_general_table은 answer_and_judge(base_model, general_rows, None)의 반환값입니다."
for score, passed in zip(base_general_table["점수"], base_general_table["통과"]):
    assert 0 <= score <= 1 and bool(passed) == (score == 1), "base_general_table의 통과 열이 점수 1인지와 다릅니다."
assert math.isfinite(base_general_ppl) and base_general_ppl > 1, f"base_general_ppl이 {base_general_ppl}입니다. perplexity(base_model, general_texts)의 반환값을 담으세요."

assert isinstance(base_tables, dict) and list(base_tables) == ["train", "domain", "refusal", "general", "ppl"], f"base_tables의 키가 {list(base_tables)}입니다. 키는 train, domain, refusal, general, ppl 순서입니다."
assert base_tables["train"] is base_train_table and base_tables["domain"] is base_domain_table, "base_tables의 train, domain에는 1번 테이블을 그대로 담습니다."
assert base_tables["refusal"] is base_refusal_table and base_tables["general"] is base_general_table and base_tables["ppl"] == base_general_ppl, "base_tables의 refusal, general, ppl에는 2번, 3번 값을 그대로 담습니다."

assert isinstance(model_metrics, dict) and list(model_metrics) == ["Base"], f"model_metrics의 키가 {list(model_metrics)}입니다. 키 Base 하나로 시작합니다."
metrics = model_metrics["Base"]
assert list(metrics) == list(METRIC_NAMES), "model_metrics[\"Base\"]는 table_metrics(base_tables)가 돌려준 사전입니다."

# 지표 = 테이블의 통과·거절 비율. 비율 * 문항 수 = 개수
for name, table, column in [("train_acc", base_train_table, "통과"), ("over_refusal_rate", base_domain_table, "거절"),
                            ("refusal_rate", base_refusal_table, "거절"), ("general_acc", base_general_table, "통과")]:
    count = 0
    for value in table[column]:
        count = count + int(value)
    assert abs(metrics[name] * len(table) - count) < 1e-6, f"model_metrics[\"Base\"]의 {name}이 테이블의 {column} 비율과 다릅니다. base_tables에 알맞은 테이블을 담았는지 확인하세요."
assert metrics["general_ppl"] == base_general_ppl, "model_metrics[\"Base\"]의 general_ppl이 base_general_ppl과 다릅니다."
assert list(forgetting_table.index) == ["Base"] and list(forgetting_table.columns) == list(METRIC_NAMES.values()), "forgetting_table은 pd.DataFrame(model_metrics).T.rename(columns=METRIC_NAMES)입니다."
print("✅ 통과!")

## 4. 에폭별 체크포인트 평가

**배경**: 교안 02의 에폭별 체크포인트 평가와 같은 방법으로, 과제 01이 lr 5e-6로 4에폭을 학습하며 `output/commerce_epochs`에 저장한 체크포인트마다 지표 일곱 개를 계산합니다. Tuned를 고르는 규칙(5번)은 교안 02처럼 결과를 보기 전에 정해 두었고, 결과를 본 뒤에 바꾸지 않습니다.

**요구사항**
- (1) `output_dir / "commerce_epochs"`(**`epoch_dir`**)의 `checkpoint-*` 폴더 이름에서 갱신 횟수를 정수로 꺼내 작은 순서로 정렬한 리스트 **`checkpoint_steps`** 를 만듭니다.
- (2) 체크포인트마다 폴더를 `AutoModelForCausalLM.from_pretrained`로 불러(float32, `device`) **`checkpoint_model`** 에 담고, `evaluate_model`이 돌려준 채점 사전을 **`epoch_tables[step]`** 에, 그 사전의 `table_metrics`를 **`epoch_metrics[step]`** 에, 이름표 `f"{epoch}에폭(step {step})"`(epoch는 1부터)를 **`epoch_labels[step]`** 에 담은 뒤, 같은 지표를 `model_metrics[이름표]`로 더합니다. 다 쓴 모델은 지우고 GPU 메모리를 비웁니다.
- (3) `forgetting_table`을 3번과 같은 식으로 다시 만듭니다.

**확인 기준**
- `checkpoint_steps`는 [34, 68, 102, 136]입니다(한 에폭 갱신 횟수 34 * 에폭).
- `epoch_tables`, `epoch_metrics`, `epoch_labels`의 키는 `checkpoint_steps`이고, `epoch_tables[step]`의 키는 `"train"`, `"domain"`, `"refusal"`, `"general"`, `"ppl"`입니다.
- `model_metrics`와 `forgetting_table`의 행은 Base, 에폭 이름표 순서이고, `checkpoint_model` 이름은 남아 있지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 에폭별 체크포인트 평가 셀과 같습니다. 폴더 이름을 정수로 바꿔 정렬하고, 체크포인트마다 불러 evaluate_model과 table_metrics를 부른 뒤 지웁니다.

세부구현:
1. epoch_dir.glob("checkpoint-*")의 path.name을 "-"로 나눈 뒤쪽이 갱신 횟수입니다. int로 바꿔 리스트에 더한 뒤 .sort()합니다.
2. enumerate(checkpoint_steps, start=1)이 에폭 번호와 갱신 횟수를 함께 돌려줍니다.
3. 정리는 del, gc.collect(), torch.cuda.empty_cache() 순서입니다.
```

</details>

In [ ]:
# [작성] (1) 체크포인트 목록을 만들고, (2) 체크포인트마다 채점 테이블과 지표 일곱 개를 계산한 뒤, (3) 지표 테이블을 다시 만드세요.
import gc

# (1) 과제 01이 에폭마다 저장한 체크포인트(폴더 이름 checkpoint-<갱신 횟수>)
epoch_dir = output_dir / "commerce_epochs"
checkpoint_steps = []
for path in epoch_dir.glob("checkpoint-*"):
    ...
checkpoint_steps.sort()
print("체크포인트 갱신 횟수:", checkpoint_steps)

# (2) 체크포인트마다 모델을 불러 채점 테이블과 지표 일곱 개를 계산하고, 다 쓴 모델은 지워 GPU 메모리를 비웁니다
epoch_tables = {}
epoch_metrics = {}
epoch_labels = {}
for epoch, step in enumerate(checkpoint_steps, start=1):
    checkpoint_model = ...
    ...

# (3) 지표 테이블에 에폭 행이 더해집니다
forgetting_table = ...
display(forgetting_table.round(3))

In [ ]:
# [자가채점] 체크포인트 목록, 에폭별 채점 사전과 지표, 지표 테이블의 행 순서, 지운 모델을 확인합니다
assert isinstance(checkpoint_steps, list) and checkpoint_steps == [34, 68, 102, 136], f"checkpoint_steps가 {checkpoint_steps}입니다(기대값 [34, 68, 102, 136]). 과제 01의 7번(에폭 늘리기)을 끝까지 실행했는지, 폴더 이름의 갱신 횟수를 int로 바꿔 정렬했는지 확인하세요."
assert list(epoch_tables) == checkpoint_steps and list(epoch_metrics) == checkpoint_steps and list(epoch_labels) == checkpoint_steps, "epoch_tables, epoch_metrics, epoch_labels의 키는 checkpoint_steps의 갱신 횟수입니다."
expected_labels = ["Base"]
for epoch, step in enumerate(checkpoint_steps, start=1):
    assert epoch_labels[step] == f"{epoch}에폭(step {step})", f"epoch_labels[{step}]이 {epoch_labels[step]}입니다. f\"{{epoch}}에폭(step {{step}})\" 모양으로 담으세요."
    expected_labels.append(epoch_labels[step])
    assert list(epoch_tables[step]) == ["train", "domain", "refusal", "general", "ppl"], f"epoch_tables[{step}]은 evaluate_model이 돌려준 사전입니다."
    recomputed = table_metrics(epoch_tables[step])
    for name in METRIC_NAMES:
        assert abs(epoch_metrics[step][name] - recomputed[name]) < 1e-9, f"epoch_metrics[{step}]의 {name}이 table_metrics(epoch_tables[{step}])와 다릅니다."
    assert math.isfinite(epoch_metrics[step]["general_ppl"]) and epoch_metrics[step]["general_ppl"] > 1, f"epoch_metrics[{step}]의 general_ppl이 {epoch_metrics[step]['general_ppl']}입니다."
    assert model_metrics[epoch_labels[step]] == epoch_metrics[step], f"model_metrics[\"{epoch_labels[step]}\"]에 epoch_metrics[{step}]을 담으세요."
assert list(model_metrics) == expected_labels, f"model_metrics의 키가 {list(model_metrics)}입니다. Base 뒤에 에폭 이름표를 차례로 더하세요."
assert list(forgetting_table.index) == expected_labels, "forgetting_table을 pd.DataFrame(model_metrics).T.rename(columns=METRIC_NAMES)로 다시 만드세요."
assert "checkpoint_model" not in globals(), "다 쓴 checkpoint_model을 del로 지워 GPU 메모리를 비우세요."
print("✅ 통과!")

## 5. Tuned 고르기

**배경**: 교안 02의 Tuned 고르기와 같은 규칙으로, 체크포인트 가운데 비교 기준이 될 하나를 판사가 채점한 정답률로 고르고 Tuned라고 부릅니다.

**고르는 규칙** (교안 02와 같습니다)
1. 학습 데이터 정답률이 `TRAIN_ACC_MIN`(0.8, 40문항 가운데 32문항) 이상인 체크포인트만 후보로 둡니다.
2. 후보 가운데 도메인 통과 수(재현 + 적용)가 가장 많은 체크포인트를 고릅니다. 통과 수가 같으면 앞 에폭을 고릅니다.
3. 후보가 하나도 없으면 학습 데이터 정답률이 가장 높은 체크포인트를 고릅니다(같으면 앞 에폭).

**요구사항**
- (1) 체크포인트마다 도메인 채점 테이블(`epoch_tables[step]["domain"]`)의 `"통과"` 수를 정수로 센 사전 **`domain_passes`**(키는 갱신 횟수)를 만듭니다.
- (2) 위 규칙을 for 문과 if 문으로 따라 고른 체크포인트의 갱신 횟수를 **`tuned_step`** 에 담습니다.
- (3) 고른 체크포인트의 에폭 번호(`checkpoint_steps`에서의 자리 + 1)를 **`tuned_epoch`** 에 담습니다. 7번에서 MLflow 태그로 남깁니다.
- (4) 고른 체크포인트의 이름표 `epoch_labels[tuned_step]`을 `f"Tuned {tuned_epoch}에폭(step {tuned_step})"`로 바꾸고, `model_metrics`를 Base와 에폭 행(새 이름표, `checkpoint_steps` 순서)으로 다시 담습니다. 뒤 문항의 테이블은 모두 이 이름표를 씁니다.
- (5) `model_metrics`에서 Base와 Tuned 두 행만 고른 **`tuned_compare_table`** 을 만듭니다(열은 `METRIC_NAMES`의 테이블 이름).

**확인 기준**
- `domain_passes`의 키는 `checkpoint_steps`이고 값은 그 도메인 테이블의 통과 수입니다.
- `tuned_step`은 위 규칙을 지킵니다(후보 가운데 통과 수가 가장 많고, 같은 통과 수의 앞 에폭이 없음).
- `tuned_epoch`은 `checkpoint_steps`에서 `tuned_step`의 자리 + 1입니다.
- `epoch_labels[tuned_step]`은 `"Tuned "`로 시작하는 위 모양이고 나머지 이름표는 4번 그대로입니다. `model_metrics`의 키는 `"Base"`와 `checkpoint_steps` 순서의 이름표이고 값은 `epoch_metrics[step]`입니다.
- `tuned_compare_table`의 행은 `"Base"`, `epoch_labels[tuned_step]`이고, Tuned 행은 `epoch_metrics[tuned_step]`입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 Tuned 고르기 셀과 같은 순서입니다. 체크포인트를 앞 에폭부터 보며, 후보이고 지금까지 고른 것보다 통과 수가 많을 때만 바꾸면 같을 때 앞 에폭이 남습니다.

세부구현:
1. 통과 수는 통과 열의 sum()을 int로 바꾼 값입니다.
2. 아직 고르지 않았음은 tuned_step = None으로 시작해 tuned_step is None으로 확인합니다.
3. 규칙 3은 첫 체크포인트로 시작해 학습 데이터 정답률이 더 높을 때만 바꿉니다.
4. 리스트.index(값)은 그 값의 자리(0부터)입니다.
5. model_metrics는 {"Base": model_metrics["Base"]}로 새로 시작해 for 문으로 에폭 행을 다시 더합니다.
6. pd.DataFrame(model_metrics).T.loc[[행 이름 목록]]으로 원하는 행만 고릅니다.
```

</details>

In [ ]:
# [작성] (1) 도메인 통과 수를 세고, (2) 규칙으로 Tuned를 고른 뒤, (3) 에폭 번호를 구하고, (4) 이름표에 Tuned를 붙여 (5) Base와 비교하세요.

# 학습 데이터 정답률이 이 값 이상인 체크포인트만 Tuned 후보로 둡니다
TRAIN_ACC_MIN = 0.8

# (1) 체크포인트마다 도메인 통과 수(재현 + 적용)를 셉니다. 평가셋 문항은 모두 재현이나 적용입니다
domain_passes = {}
selection_rows = []
for step in checkpoint_steps:
    domain_passes[step] = ...
    selection_rows.append({"체크포인트": epoch_labels[step], "학습 데이터 정답률": epoch_metrics[step]["train_acc"],
                           "후보": epoch_metrics[step]["train_acc"] >= TRAIN_ACC_MIN, "도메인 통과 수": domain_passes[step]})
display(pd.DataFrame(selection_rows).round(3))

# (2) 규칙 1, 2: 후보 가운데 도메인 통과 수가 가장 많은 체크포인트. 앞 에폭부터 보며 더 많을 때만 바꾸므로 같으면 앞 에폭이 남습니다
tuned_step = None
for step in checkpoint_steps:
    ...

# 규칙 3: 후보가 없으면 학습 데이터 정답률이 가장 높은 체크포인트(같으면 앞 에폭)
if tuned_step is None:
    ...

# (3) 고른 체크포인트의 에폭 번호는 7번에서 MLflow 태그로 남깁니다
tuned_epoch = ...

# (4) 고른 체크포인트의 이름표 앞에 Tuned 를 붙이고, model_metrics 의 에폭 행을 새 이름표로 다시 담습니다(행 순서는 그대로)
epoch_labels[tuned_step] = ...
model_metrics = {"Base": model_metrics["Base"]}
for step in checkpoint_steps:
    ...
print("고른 체크포인트:", epoch_labels[tuned_step], "| 에폭 수:", tuned_epoch, "| 폴더:", epoch_dir / f"checkpoint-{tuned_step}")

# (5) model_metrics 에서 Base와 Tuned 두 행만 골라 봅니다
tuned_compare_table = ...
display(tuned_compare_table.round(3))

# 평가셋: 같은 질문의 Base 답과 Tuned 답, 판정을 나란히 봅니다(채점 테이블에서 꺼내므로 다시 생성하지 않습니다)
tuned_domain_table = epoch_tables[tuned_step]["domain"]
display(pd.DataFrame({
    "종류": base_domain_table["종류"], "질문": base_domain_table["질문"],
    "Base 답변": base_domain_table["답변"], "Base 통과": base_domain_table["통과"],
    "Tuned 답변": tuned_domain_table["답변"], "Tuned 통과": tuned_domain_table["통과"],
    "Tuned 판정 이유": tuned_domain_table["이유"],
}))

In [ ]:
# [자가채점] 도메인 통과 수, 고르는 규칙의 불변식, 에폭 번호, 비교 테이블을 확인합니다(어느 체크포인트가 뽑혀야 하는지는 보지 않습니다)
assert TRAIN_ACC_MIN == 0.8, "TRAIN_ACC_MIN은 0.8으로 둡니다(교안 02와 같은 값)."
assert isinstance(domain_passes, dict) and list(domain_passes) == checkpoint_steps, "domain_passes의 키는 checkpoint_steps의 갱신 횟수입니다."
for step in checkpoint_steps:
    passed_count = 0
    for passed in epoch_tables[step]["domain"]["통과"]:
        passed_count = passed_count + int(passed)
    assert domain_passes[step] == passed_count, f"domain_passes[{step}]가 {domain_passes[step]}입니다. 그 체크포인트 도메인 테이블의 통과 수는 {passed_count}입니다."

assert tuned_step in checkpoint_steps, f"tuned_step이 {tuned_step}입니다. checkpoint_steps 가운데 하나를 담으세요."
candidates = []
for step in checkpoint_steps:
    if epoch_metrics[step]["train_acc"] >= TRAIN_ACC_MIN:
        candidates.append(step)
if len(candidates) > 0:
    # 규칙 1, 2: 후보이고, 통과 수가 가장 많고, 같으면 앞 에폭
    assert tuned_step in candidates, f"후보(학습 데이터 정답률 {TRAIN_ACC_MIN} 이상)는 {candidates}인데 tuned_step이 {tuned_step}입니다. 규칙 1을 확인하세요."
    for step in candidates:
        assert domain_passes[step] <= domain_passes[tuned_step], f"후보 {step}의 도메인 통과 수가 더 많습니다. 규칙 2를 확인하세요."
        if step < tuned_step:
            assert domain_passes[step] < domain_passes[tuned_step], f"앞 에폭 후보 {step}의 통과 수가 같습니다. 같으면 앞 에폭을 고릅니다."
else:
    # 규칙 3: 학습 데이터 정답률이 가장 높고, 같으면 앞 에폭
    for step in checkpoint_steps:
        assert epoch_metrics[step]["train_acc"] <= epoch_metrics[tuned_step]["train_acc"], f"후보가 없을 때는 학습 데이터 정답률이 가장 높은 체크포인트를 고릅니다. {step}이 더 높습니다."
        if step < tuned_step:
            assert epoch_metrics[step]["train_acc"] < epoch_metrics[tuned_step]["train_acc"], f"앞 에폭 {step}의 학습 데이터 정답률이 같습니다. 같으면 앞 에폭을 고릅니다."

assert tuned_epoch == checkpoint_steps.index(tuned_step) + 1, f"tuned_epoch이 {tuned_epoch}입니다. checkpoint_steps에서 tuned_step의 자리 + 1을 담으세요."

# 이름표: 고른 체크포인트만 앞에 Tuned, model_metrics 는 Base 와 새 이름표 순서
expected_labels = ["Base"]
for epoch, step in enumerate(checkpoint_steps, start=1):
    if step == tuned_step:
        assert epoch_labels[step] == f"Tuned {epoch}에폭(step {step})", f"epoch_labels[{step}]이 {epoch_labels[step]}입니다. f\"Tuned {{tuned_epoch}}에폭(step {{tuned_step}})\" 모양으로 바꾸세요."
    else:
        assert epoch_labels[step] == f"{epoch}에폭(step {step})", f"epoch_labels[{step}]이 {epoch_labels[step]}입니다. 고르지 않은 체크포인트의 이름표는 4번 그대로 둡니다."
    expected_labels.append(epoch_labels[step])
assert list(model_metrics) == expected_labels, f"model_metrics의 키가 {list(model_metrics)}입니다. Base 뒤에 checkpoint_steps 순서의 새 이름표로 다시 담으세요."
for step in checkpoint_steps:
    assert model_metrics[epoch_labels[step]] == epoch_metrics[step], f"model_metrics[\"{epoch_labels[step]}\"]에 epoch_metrics[{step}]을 담으세요."

assert isinstance(tuned_compare_table, pd.DataFrame) and list(tuned_compare_table.index) == ["Base", epoch_labels[tuned_step]], f"tuned_compare_table의 행이 {list(tuned_compare_table.index)}입니다. model_metrics에서 Base와 epoch_labels[tuned_step] 두 행을 고르세요."
assert list(tuned_compare_table.columns) == list(METRIC_NAMES.values()), "tuned_compare_table의 열은 METRIC_NAMES의 테이블 이름 일곱 개입니다(.rename(columns=METRIC_NAMES))."
for name, column in METRIC_NAMES.items():
    assert abs(tuned_compare_table.loc[epoch_labels[tuned_step], column] - epoch_metrics[tuned_step][name]) < 1e-9, "tuned_compare_table의 Tuned 행은 epoch_metrics[tuned_step]입니다."
print("✅ 통과!")

## 6. 가중치 보간 α 두 개

**배경**: 교안 02의 가중치 보간과 같은 방법으로, Base와 마지막 에폭 체크포인트의 가중치를 이름이 같은 텐서끼리 비율 α로 섞어 다시 학습하지 않고 새 모델을 만듭니다. α는 교안 02와 같은 두 값(0.5, 0.75)이고, α가 클수록 학습한 것을 더 남기는 대신 일반 능력은 덜 되찾는 맞바꿈을 같은 지표 일곱 개로 봅니다.

**요구사항**
- (1) 마지막 체크포인트의 갱신 횟수 **`last_step`**(`checkpoint_steps`의 마지막 값)의 폴더를 float32로 불러 `device`에 올린 **`last_model`** 을 만들고, `base_model.state_dict()`, `last_model.state_dict()`를 **`base_state`**, **`last_state`** 에, 섞는 비율 목록 **`interp_alphas`**([0.5, 0.75])를 담습니다.
- (2) 섞은 가중치를 넣을 모델 **`interp_model`**(`MODEL_NAME`, float32, `device`)을 하나 만듭니다.
- (3) `interp_alphas`의 α마다 섞은 가중치 사전 `mixed_state`(만드는 줄은 채워 두었습니다)를 `interp_model.load_state_dict`로 넣고, 이름표 `interp_label`(`f"보간 α={interp_alpha}"`)로 `model_metrics[interp_label]`에 `table_metrics(evaluate_model(interp_model))`을 담은 뒤, 이름표를 리스트 **`interp_labels`** 에 더합니다.
- (4) 비교 테이블 `interp_compare_table`(Base, 보간 모델, 마지막 체크포인트를 α 순서로 고른 테이블)을 만드는 줄은 채워 두었습니다.

**확인 기준**
- `last_step`은 `checkpoint_steps`의 가장 큰 값이고, `last_model`은 float32이며, `interp_alphas`는 [0.5, 0.75]입니다.
- 반복이 끝난 뒤 `interp_model`의 가중치는 마지막 α로 섞은 `(1 - α) * base_state + α * last_state`와 같습니다(첫 실수 텐서 하나로 확인).
- `interp_labels`는 α마다 `f"보간 α={interp_alpha}"`이고, `model_metrics`의 마지막 두 키가 `interp_labels`이며, 그 값은 지표 일곱 개(비율은 0~1, perplexity는 1보다 큰 유한한 값)입니다.
- `interp_compare_table`의 행은 `"Base"`, `interp_labels`, `epoch_labels[last_step]` 순서입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 두 모델의 가중치와 보간 모델의 지표 셀과 같습니다. 두 모델의 가중치 사전을 꺼내고, 모델 하나에 α마다 섞은 사전을 넣어 평가합니다.

세부구현:
1. state_dict()는 가중치 이름에서 텐서로 가는 사전입니다.
2. load_state_dict에 섞은 사전을 넘기면 interp_model의 가중치가 바뀝니다. 섞는 식은 새 텐서를 만들므로 두 모델의 가중치는 바뀌지 않습니다.
3. 이름표 f-string은 지문의 모양을 그대로 씁니다. 반복 안에서 model_metrics에 담은 뒤 interp_labels.append로 더합니다.
```

</details>

In [ ]:
# [작성] (1) 마지막 체크포인트와 두 가중치 사전, α 목록을 준비하고, (2) 보간 모델을 만든 뒤, (3) α마다 섞은 가중치로 지표를 계산하세요.

# (1) 마지막 에폭 체크포인트(갱신 횟수가 가장 큰 폴더)를 불러옵니다
last_step = ...
last_model = ...
print("섞을 체크포인트:", epoch_labels[last_step])

# 두 모델의 가중치 사전: 가중치 이름 -> 텐서
base_state = ...
last_state = ...

# 섞을 마지막 체크포인트의 비율 α 두 개(0은 Base, 1은 마지막 체크포인트)
interp_alphas = ...

# (2) 섞은 가중치를 넣을 모델을 만듭니다
interp_model = ...

# (3) α마다 이름이 같은 텐서끼리 (1 - α) * Base + α * 마지막 체크포인트로 섞어 넣고 지표 일곱 개를 계산합니다
interp_labels = []
for interp_alpha in interp_alphas:
    mixed_state = {}
    for name in base_state:
        mixed_state[name] = (1 - interp_alpha) * base_state[name] + interp_alpha * last_state[name]
    ...

    # 보간 모델의 이름표와 지표 일곱 개
    interp_label = ...
    model_metrics[interp_label] = ...
    ...

# (4) model_metrics 에서 Base(α = 0), 보간 모델, 마지막 체크포인트(α = 1)를 α 순서로 골라 봅니다
interp_rows = ["Base"] + interp_labels + [epoch_labels[last_step]]
interp_compare_table = pd.DataFrame(model_metrics).T.loc[interp_rows].rename(columns=METRIC_NAMES)
display(interp_compare_table.round(3))

In [ ]:
# [자가채점] 섞을 체크포인트와 비율, 섞은 가중치, 보간 지표와 비교 테이블을 확인합니다
assert last_step == max(checkpoint_steps), f"last_step이 {last_step}입니다. checkpoint_steps의 가장 큰 값(마지막 에폭)을 담으세요."
assert interp_alphas == [0.5, 0.75], f"interp_alphas가 {interp_alphas}입니다. [0.5, 0.75]로 두세요."
assert next(last_model.parameters()).dtype == torch.float32, "last_model은 마지막 체크포인트 폴더를 dtype=torch.float32로 불러옵니다."

# 반복이 끝나면 interp_model 에는 마지막 α로 섞은 가중치가 들어 있어야 합니다(부동소수 계산 순서 차이만큼은 허용)
last_alpha = interp_alphas[-1]
interp_state = interp_model.state_dict()
for name in base_state:
    if base_state[name].is_floating_point():
        expected_weight = (1 - last_alpha) * base_state[name] + last_alpha * last_state[name]
        assert torch.allclose(interp_state[name], expected_weight, atol=1e-6), f"interp_model의 {name}이 (1 - α) * Base + α * 마지막 체크포인트(α = {last_alpha})와 다릅니다. α마다 mixed_state를 load_state_dict로 넣었는지 확인하세요."
        break

# α마다 이름표와 지표 일곱 개
expected_interp_labels = []
for interp_alpha in interp_alphas:
    expected_interp_labels.append(f"보간 α={interp_alpha}")
assert interp_labels == expected_interp_labels, f"interp_labels가 {interp_labels}입니다. α마다 f\"보간 α={{interp_alpha}}\"를 차례로 더하세요."
assert list(model_metrics)[-len(interp_labels):] == interp_labels, f"model_metrics의 마지막 키가 {list(model_metrics)[-len(interp_labels):]}입니다. α마다 model_metrics[interp_label]에 담으세요."
for interp_label in interp_labels:
    assert list(model_metrics[interp_label]) == list(METRIC_NAMES), f"model_metrics[\"{interp_label}\"]에 table_metrics(evaluate_model(interp_model))을 담으세요."
    for name in ["train_acc", "domain_recall_acc", "domain_apply_acc", "over_refusal_rate", "refusal_rate", "general_acc"]:
        assert 0 <= model_metrics[interp_label][name] <= 1, f"{interp_label}의 {name}가 {model_metrics[interp_label][name]}입니다."
    assert math.isfinite(model_metrics[interp_label]["general_ppl"]) and model_metrics[interp_label]["general_ppl"] > 1, f"{interp_label}의 general_ppl이 1보다 큰 유한한 값이어야 합니다."
assert list(interp_compare_table.index) == ["Base"] + interp_labels + [epoch_labels[last_step]], f"interp_compare_table의 행이 {list(interp_compare_table.index)}입니다. 채워 둔 줄을 그대로 실행하세요."
print("✅ 통과!")

## 7. MLflow 기록과 결정 테이블

**배경**: 교안 02의 기록과 결정과 같은 방법입니다. 평가 지표는 MLflow에 기록으로 남기고, 모델은 결정 테이블로 고릅니다. 학습한 에폭 run은 `mlflow.start_run(run_id=...)`로 다시 열어 평가 지표를 더하고, 학습한 run이 없는 Base와 보간 모델(α마다 하나)은 새 run으로 남깁니다. 손실은 학습 기록이라 결정 테이블에 넣지 않습니다.

**요구사항**
- (1) 실험 `commerce-sft`의 run 목록(최근 순)에서 run 이름이 `"epochs4_lr_5e-6"`인 가장 최근 run의 `run_id`를 **`epoch_run_id`** 에 담습니다.
- (2) 에폭 run을 다시 열어 `epoch_metrics[step]`을 `step=step`으로 기록한 뒤 태그 `tuned_step`, `tuned_epoch`를 남깁니다.
- (3) run 이름 `"base"`로 새 run을 만들어 `model_name` 파라미터와 Base 지표를 기록하고, `interp_alphas`의 α마다 run 이름 `f"interp_{interp_alpha}"`로 새 run을 만들어 파라미터 `alpha`, `mixed_checkpoint`(섞은 체크포인트 폴더 경로 문자열)와 그 보간 모델의 지표(`model_metrics[f"보간 α={interp_alpha}"]`)를 기록합니다.
- (4) `model_metrics`의 모든 모델을 행으로 둔 결정 테이블 **`summary_table`** 을 `pd.DataFrame(model_metrics).T.rename(columns=METRIC_NAMES)` 한 줄로 만듭니다. Tuned 이름표는 5번에서 이미 붙였습니다.

**확인 기준**
- `epoch_run_id`는 run 이름이 `epochs4_lr_5e-6`인 run이고, 그 run에 `checkpoint_steps`마다 step이 붙은 지표와 태그(`tuned_step`, `tuned_epoch`)가 있습니다.
- 새 run `base`와 α마다의 보간 run(`interp_<α>`)이 있고, 가장 최근 그 이름의 run에 그 모델의 `general_acc`가 기록되어 있습니다.
- `summary_table`의 행은 `model_metrics`의 키 순서 그대로이고, 열은 지표 일곱 개입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 MLflow에 평가 지표 기록하기 셀과 서비스에 쓸 모델 고르기 셀과 같습니다.

세부구현:
1. search_runs 결과에서 "tags.mlflow.runName" 열이 같은 행의 "run_id"를 .iloc[0]으로 꺼냅니다(최근 run이 맨 위).
2. log_metrics에 step=을 주면 같은 지표 이름에 step마다 값이 쌓입니다. 태그는 set_tags에 사전으로 넘깁니다.
3. 보간 run은 for 문으로 interp_alphas를 돌며 반복마다 mlflow.start_run을 엽니다.
```

</details>

In [ ]:
# [작성] (1)~(3) 평가 지표를 MLflow run에 기록하고, (4) 결정 테이블을 만드세요.

# (1) 같은 실험의 run 목록에서 run 이름으로 run_id 를 찾습니다. 다시 실행해 같은 이름이 여러 개면 가장 최근 것(맨 위)을 씁니다
runs_table = mlflow.search_runs(experiment_names=["commerce-sft"], order_by=["start_time DESC"])
epoch_run_id = ...
print("찾은 run:", epoch_run_id)

# (2) 에폭 run 을 다시 열고, 체크포인트의 갱신 횟수를 step 으로 넣어 MLflow 화면에서 학습 기록과 같은 가로축에 놓습니다
...

# (3) Base는 학습한 run 이 없으므로 새 run 을 만듭니다
...

# 보간 모델은 α마다 새 run 을 만들고, α와 섞은 체크포인트를 남깁니다
for interp_alpha in interp_alphas:
    ...

# (4) 모든 모델의 지표 일곱 개. 고른 체크포인트의 이름표에는 5번에서 Tuned 를 붙였습니다
summary_table = ...
display(summary_table.round(3))

In [ ]:
# [자가채점] MLflow에 기록한 지표, 태그, 새 run, 결정 테이블의 행과 열을 확인합니다
epoch_run = mlflow.get_run(epoch_run_id)
assert epoch_run.info.run_name == "epochs4_lr_5e-6", f"epoch_run_id는 run 이름이 {epoch_run.info.run_name}인 run입니다. run 이름이 epochs4_lr_5e-6인 run의 run_id를 담으세요."
logged_steps = []
for metric in mlflow.MlflowClient().get_metric_history(epoch_run_id, "train_acc"):
    logged_steps.append(metric.step)
for step in checkpoint_steps:
    assert step in logged_steps, f"에폭 run에 step {step}의 지표가 없습니다. log_metrics(epoch_metrics[step], step=step)으로 기록하세요."
assert epoch_run.data.tags.get("tuned_step") == str(tuned_step) and epoch_run.data.tags.get("tuned_epoch") == str(tuned_epoch), "에폭 run에 태그 tuned_step, tuned_epoch를 set_tags로 남기세요."

runs_after = mlflow.search_runs(experiment_names=["commerce-sft"], order_by=["start_time DESC"])
new_runs = [("base", "Base")]
for interp_alpha in interp_alphas:
    new_runs.append((f"interp_{interp_alpha}", f"보간 α={interp_alpha}"))
for run_name, model_label in new_runs:
    found = runs_after[runs_after["tags.mlflow.runName"] == run_name]
    assert len(found) > 0, f"run {run_name}이 없습니다. mlflow.start_run(run_name=...)으로 새 run을 만드세요."
    assert abs(found.iloc[0]["metrics.general_acc"] - model_metrics[model_label]["general_acc"]) < 1e-6, f"run {run_name}에 {model_label}의 지표를 log_metrics로 기록하세요."

assert isinstance(summary_table, pd.DataFrame) and list(summary_table.index) == list(model_metrics), f"summary_table의 행이 {list(summary_table.index)}입니다. pd.DataFrame(model_metrics).T로 model_metrics 순서 그대로 두세요."
assert list(summary_table.columns) == list(METRIC_NAMES.values()), f"summary_table의 열이 {list(summary_table.columns)}입니다. 열은 지표 일곱 개뿐입니다(손실은 학습 기록이라 넣지 않습니다)."
print("✅ 통과!")

## 8. 고른 모델 저장

**배경**: 교안 02의 고른 모델 저장과 같은 방법으로, 5번에서 고른 Tuned 체크포인트를 최종 모델 폴더에 모델과 토크나이저를 함께 저장합니다. 서빙과 다음 시간 비교는 이 폴더를 쓰므로 학습은 다시 하지 않습니다(디스크 약 3GB).

**요구사항**
- (1) 고른 Tuned 체크포인트 폴더(`epoch_dir / f"checkpoint-{tuned_step}"`)를 float32로 불러 `device`에 올린 **`final_model`** 을 만들고, 평가셋 첫 질문(`eval_rows[0]["question"]`)을 `generate_replies`로 `SYSTEM_PROMPT_COMMERCE`와 함께 물어 답을 출력합니다.
- (2) 최종 모델 폴더 **`final_dir`**(`output_dir / "commerce_sft_final"`)에 `final_model`과 `tokenizer`를 `save_pretrained`로 저장합니다.
- (3) `final_dir`의 파일마다 키가 `"파일"`(파일 이름), `"크기(MB)"`(소수 첫째 자리)인 사전을 담은 리스트 **`final_saved_files`** 를 만들어 테이블로 봅니다.
- (4) 다 쓴 `final_model`을 del로 지우고 GPU 메모리를 비웁니다.

**확인 기준**
- `final_dir`는 `output_dir / "commerce_sft_final"`이고, 그 안에 `config.json`, 가중치 `model.safetensors`(나뉘어 저장되면 `model.safetensors.index.json`), 토크나이저 `tokenizer.json`, `tokenizer_config.json`이 있습니다.
- 저장한 `config.json`의 `architectures`가 고른 체크포인트 폴더의 `config.json`과 같습니다.
- `final_saved_files`는 `final_dir`의 파일마다 사전 하나(키 `"파일"`, `"크기(MB)"`)입니다.
- `final_model` 이름은 남아 있지 않습니다. 답변 내용은 보지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 고른 모델 저장 셀 두 개와 같습니다. 4번처럼 체크포인트 폴더를 불러 한 번 묻고, 모델과 토크나이저를 같은 폴더에 저장한 뒤 지웁니다.

세부구현:
1. generate_replies는 질문 리스트를 받아 답 리스트를 돌려주므로, 질문 하나도 리스트로 감싸고 답은 [0]으로 꺼냅니다.
2. save_pretrained는 폴더가 없으면 만듭니다. 모델과 토크나이저에 같은 폴더를 넘깁니다.
3. sorted(final_dir.iterdir())로 파일을 이름 순으로 돌며 path.name과, path.stat().st_size를 1024 ** 2로 나눈 값을 씁니다.
4. 정리는 4번과 같이 del, gc.collect(), torch.cuda.empty_cache() 순서입니다.
```

</details>

In [ ]:
# [작성] (1) 고른 Tuned를 불러 한 번 묻고, (2) 최종 모델 폴더에 저장한 뒤, (3) 저장한 파일을 보고, (4) 모델을 지우세요.

# (1) 고른 Tuned 체크포인트를 불러옵니다
final_model = ...

# 평가셋 첫 질문을 물어, 불러온 모델이 고른 Tuned 인지 답으로 확인합니다
question = eval_rows[0]["question"]
print("질문:", question)
print("답변:", ...)

# (2) 모델과 토크나이저를 최종 모델 폴더에 저장합니다
final_dir = ...
...

# (3) 저장된 파일 이름과 크기(MB)
final_saved_files = []
for path in sorted(final_dir.iterdir()):
    ...
display(pd.DataFrame(final_saved_files))

# (4) 저장한 모델은 다 썼으므로 지워 GPU 메모리를 비웁니다
...
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# [자가채점] 최종 모델 폴더의 파일, 저장한 설정의 모델 구조, 지운 모델을 확인합니다(답변 내용은 보지 않습니다)
assert final_dir == output_dir / "commerce_sft_final", f"final_dir이 {final_dir}입니다. output_dir / \"commerce_sft_final\"에 저장하세요."
assert final_dir.is_dir(), f"{final_dir} 폴더가 없습니다. final_model.save_pretrained(final_dir)를 실행했는지 확인하세요."

# 모델 설정, 가중치, 토크나이저 파일이 한 폴더에 있어야 합니다
saved_names = []
for path in final_dir.iterdir():
    saved_names.append(path.name)
assert "config.json" in saved_names, f"{final_dir}에 config.json이 없습니다. final_model.save_pretrained(final_dir)로 저장하세요."
assert "model.safetensors" in saved_names or "model.safetensors.index.json" in saved_names, "가중치 파일(model.safetensors 또는 model.safetensors.index.json)이 없습니다. final_model.save_pretrained(final_dir)로 저장하세요."
for name in ["tokenizer.json", "tokenizer_config.json"]:
    assert name in saved_names, f"{final_dir}에 {name}이 없습니다. tokenizer.save_pretrained(final_dir)로 같은 폴더에 저장하세요."

# 저장한 모델 구조가 고른 체크포인트와 같아야 합니다
saved_config = json.loads((final_dir / "config.json").read_text(encoding="utf-8"))
checkpoint_config = json.loads((epoch_dir / f"checkpoint-{tuned_step}" / "config.json").read_text(encoding="utf-8"))
assert saved_config["architectures"] == checkpoint_config["architectures"], f"저장한 모델 구조가 {saved_config['architectures']}입니다. 고른 Tuned 체크포인트(epoch_dir / f\"checkpoint-{{tuned_step}}\")를 불러 저장하세요."

# 파일 목록 테이블과 지운 모델
assert isinstance(final_saved_files, list) and len(final_saved_files) == len(saved_names), f"final_saved_files가 {len(final_saved_files)}개입니다. final_dir의 파일마다 사전 하나를 담으세요."
for row in final_saved_files:
    assert list(row) == ["파일", "크기(MB)"], f"final_saved_files의 사전 키가 {list(row)}입니다. 키는 \"파일\", \"크기(MB)\"입니다."
assert "final_model" not in globals(), "다 쓴 final_model을 del로 지워 GPU 메모리를 비우세요."
print("✅ 통과!")